# Spiking Neural Networks with Grilly

**Biologically-inspired computing with leaky integrate-and-fire neurons.**

This notebook covers:
1. GPU/CPU detection
2. What are spiking neural networks (SNNs)?
3. The Leaky Integrate-and-Fire (LIF) neuron model
4. Building a simple SNN classifier
5. Rate coding: encoding inputs as spike trains
6. Training with surrogate gradients
7. Visualizing membrane potentials and spike rasters
8. ANN-to-SNN conversion example

Grilly provides `LIFNode`, `IFNode`, and `MultiStepContainer` in
`grilly.nn.snn` for building spiking networks.

In [ ]:
import grilly
import grilly.functional as F
import matplotlib.pyplot as plt
import numpy as np
from grilly import nn
from grilly.nn import Variable
from grilly.nn.snn import IFNode, LIFNode
from grilly.optim import Adam

# --- Backend detection ---
try:
    from grilly._bridge import is_vulkan_available
    DEVICE = "vulkan" if is_vulkan_available() else "cpu"
except (ImportError, AttributeError):
    DEVICE = "cpu"

print(f"grilly {grilly.__version__} | backend: {DEVICE}")
np.random.seed(42)

## 1. What Are Spiking Neural Networks?

Unlike traditional artificial neural networks (ANNs) that process
continuous-valued activations, spiking neural networks communicate via
discrete **spikes** (binary events). Each neuron maintains an internal
**membrane potential** that accumulates input over time. When the membrane
potential exceeds a threshold, the neuron fires a spike and resets.

### Advantages
- **Temporal coding**: information encoded in spike timing, not just rates
- **Energy efficiency**: sparse binary communication (ideal for neuromorphic hardware)
- **Biological plausibility**: closer to real neurons

### The LIF Neuron

The Leaky Integrate-and-Fire (LIF) model is the most common spiking neuron:

```
membrane potential:  V[t] = beta * V[t-1] + I[t]    (leaky integration)
spike:              S[t] = 1 if V[t] >= threshold, else 0
reset:              V[t] = V[t] - threshold * S[t]   (soft reset)
```

Where `beta` is the decay factor (0 < beta < 1) and `I[t]` is the input
current at timestep `t`.

## 2. Visualize a Single LIF Neuron

Let's observe how a single LIF neuron responds to constant input current.
We'll plot the membrane potential and the spike times.

In [ ]:
# Simulate a single LIF neuron with constant input
n_steps = 100
beta = 0.85       # Membrane decay factor
threshold = 1.0   # Spike threshold
input_current = 0.3  # Constant input

membrane = np.zeros(n_steps)
spikes = np.zeros(n_steps)
v = 0.0

for t in range(n_steps):
    # Leaky integration
    v = beta * v + input_current

    # Check threshold
    if v >= threshold:
        spikes[t] = 1.0
        v = v - threshold  # Soft reset

    membrane[t] = v

# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

axes[0].plot(membrane, color='#2c3e50', linewidth=1.5)
axes[0].axhline(y=threshold, color='#e74c3c', linestyle='--', alpha=0.7, label='Threshold')
axes[0].set_ylabel('Membrane Potential')
axes[0].set_title(f'LIF Neuron (beta={beta}, threshold={threshold}, input={input_current})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

spike_times = np.where(spikes > 0)[0]
axes[1].eventplot(spike_times, lineoffsets=0.5, linelengths=0.8, color='#e74c3c')
axes[1].set_ylabel('Spikes')
axes[1].set_xlabel('Timestep')
axes[1].set_yticks([])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Total spikes: {int(spikes.sum())}")
print(f"Spike rate  : {spikes.mean():.2f} spikes/timestep")

## 3. Rate Coding: Encoding Inputs as Spike Trains

To feed continuous data into an SNN, we use **rate coding**: each input
value is converted to a Poisson spike train whose firing rate is
proportional to the input magnitude.

In [ ]:
def rate_encode(x, n_steps=25):
    """Convert input values to spike trains using rate coding.

    Args:
        x: Input array, values in [0, 1]. Shape: (batch, features)
        n_steps: Number of timesteps to simulate

    Returns:
        Spike trains of shape (n_steps, batch, features)
    """
    # Clamp inputs to [0, 1] for valid probabilities
    x_clamped = np.clip(x, 0.0, 1.0)
    # At each timestep, fire a spike with probability = input value
    spikes = np.random.rand(n_steps, *x.shape).astype(np.float32) < x_clamped
    return spikes.astype(np.float32)

# Demonstrate rate coding on a simple 4-feature input
demo_input = np.array([[0.1, 0.4, 0.7, 1.0]], dtype=np.float32)
demo_spikes = rate_encode(demo_input, n_steps=50)

fig, axes = plt.subplots(4, 1, figsize=(10, 4), sharex=True)
for i in range(4):
    spike_times = np.where(demo_spikes[:, 0, i] > 0)[0]
    axes[i].eventplot(spike_times, lineoffsets=0.5, linelengths=0.8, color='#2c3e50')
    axes[i].set_ylabel(f'p={demo_input[0, i]:.1f}')
    axes[i].set_yticks([])

axes[-1].set_xlabel('Timestep')
axes[0].set_title('Rate Coding: Higher input value = more spikes')
plt.tight_layout()
plt.show()

## 4. Build a Simple SNN

We build a 3-layer SNN for classifying simple patterns:

```
Input (8) -> Linear(8, 64) -> LIFNode -> Linear(64, 64) -> LIFNode -> Linear(64, 4) -> Output
```

The output is the membrane potential of the final layer accumulated over
all timesteps (rate decoding).

In [ ]:
# Create a simple SNN
n_input = 8
n_hidden = 64
n_output = 4
n_steps = 25

# Linear layers (no activation -- LIF neurons provide nonlinearity)
fc1 = nn.Linear(n_input, n_hidden)
lif1 = LIFNode(beta=0.9, threshold=1.0)

fc2 = nn.Linear(n_hidden, n_hidden)
lif2 = LIFNode(beta=0.9, threshold=1.0)

fc3 = nn.Linear(n_hidden, n_output)

print("SNN Architecture:")
print(f"  Input  : {n_input} features")
print(f"  Hidden : {n_hidden} LIF neurons x 2 layers")
print(f"  Output : {n_output} classes")
print(f"  Time   : {n_steps} timesteps")

## 5. Generate Synthetic Data and Train

We create a simple 4-class classification problem and train the SNN
using surrogate gradients. The loss is computed on the accumulated
output spike counts (rate decoding).

In [ ]:
# Generate simple classification data
n_samples = 400
X_data = np.random.randn(n_samples, n_input).astype(np.float32) * 0.5
y_labels = np.random.randint(0, n_output, size=n_samples)

# Add class-specific signal so the problem is learnable
for i in range(n_samples):
    c = y_labels[i]
    X_data[i, c * 2:(c + 1) * 2] += 1.5

# Normalize to [0, 1] for rate coding
X_data = (X_data - X_data.min()) / (X_data.max() - X_data.min() + 1e-8)

# One-hot targets
y_onehot = np.zeros((n_samples, n_output), dtype=np.float32)
y_onehot[np.arange(n_samples), y_labels] = 1.0

print(f"Data shape  : {X_data.shape}")
print(f"Labels shape: {y_labels.shape}")
print(f"Classes     : {np.bincount(y_labels)}")

In [ ]:
# Training loop
all_params = list(fc1.parameters()) + list(fc2.parameters()) + list(fc3.parameters())
optimizer = Adam(all_params, lr=5e-3)

n_epochs = 80
batch_size = 64
loss_history = []

for epoch in range(n_epochs):
    # Shuffle data each epoch
    perm = np.random.permutation(n_samples)
    epoch_loss = 0.0
    n_batches = 0

    for start in range(0, n_samples, batch_size):
        idx = perm[start:start + batch_size]
        x_batch = X_data[idx]
        t_batch = y_onehot[idx]
        bs = len(idx)

        # Rate encode the input into spike trains
        spike_trains = rate_encode(x_batch, n_steps=n_steps)

        # Reset LIF membrane states
        lif1.reset()
        lif2.reset()

        # Accumulate output over timesteps (rate decoding)
        output_accum = np.zeros((bs, n_output), dtype=np.float32)

        for t in range(n_steps):
            x_t = Variable(spike_trains[t])

            # Layer 1: Linear -> LIF
            h1 = fc1(x_t)
            s1 = lif1(h1)

            # Layer 2: Linear -> LIF
            h2 = fc2(s1)
            s2 = lif2(h2)

            # Layer 3: Linear (output accumulation)
            out = fc3(s2)
            output_accum += out.data

        # Normalize by timesteps
        output_var = Variable(output_accum / n_steps)
        target_var = Variable(t_batch)

        # MSE loss on accumulated output
        diff = output_var - target_var
        loss = (diff * diff).sum() / bs

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += float(loss.data)
        n_batches += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    loss_history.append(avg_loss)

    if (epoch + 1) % 20 == 0 or epoch == 0:
        # Compute accuracy
        spike_all = rate_encode(X_data, n_steps=n_steps)
        lif1.reset()
        lif2.reset()
        out_accum = np.zeros((n_samples, n_output), dtype=np.float32)
        for t in range(n_steps):
            x_t = Variable(spike_all[t])
            s1 = lif1(fc1(x_t))
            s2 = lif2(fc2(s1))
            out_accum += fc3(s2).data
        preds = np.argmax(out_accum, axis=1)
        acc = np.mean(preds == y_labels) * 100
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.4f} | Accuracy: {acc:.1f}%")

print("\nTraining complete.")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(loss_history, linewidth=1.5, color='#8e44ad')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('SNN Training Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Visualize Membrane Potentials and Spike Rasters

Let's feed a single sample through the trained SNN and record the
membrane potentials and spike events at each layer.

In [ ]:
# Pick a single sample
sample_idx = 0
x_sample = X_data[sample_idx:sample_idx+1]
spike_input = rate_encode(x_sample, n_steps=50)  # More timesteps for visualization

# Record membrane potentials and spikes
lif1.reset()
lif2.reset()

n_vis_steps = 50
n_neurons_to_show = 16  # Show first 16 neurons from each layer

mem1_history = np.zeros((n_vis_steps, n_neurons_to_show))
spk1_history = np.zeros((n_vis_steps, n_neurons_to_show))
mem2_history = np.zeros((n_vis_steps, n_neurons_to_show))
spk2_history = np.zeros((n_vis_steps, n_neurons_to_show))

for t in range(n_vis_steps):
    x_t = Variable(spike_input[t])
    h1 = fc1(x_t)
    s1 = lif1(h1)
    h2 = fc2(s1)
    s2 = lif2(h2)

    # Record membrane and spikes (access internal state)
    if hasattr(lif1, 'mem'):
        mem1_history[t] = lif1.mem.data[0, :n_neurons_to_show] if lif1.mem.data.ndim > 1 else lif1.mem.data[:n_neurons_to_show]
    spk1_history[t] = s1.data[0, :n_neurons_to_show] if s1.data.ndim > 1 else s1.data[:n_neurons_to_show]
    if hasattr(lif2, 'mem'):
        mem2_history[t] = lif2.mem.data[0, :n_neurons_to_show] if lif2.mem.data.ndim > 1 else lif2.mem.data[:n_neurons_to_show]
    spk2_history[t] = s2.data[0, :n_neurons_to_show] if s2.data.ndim > 1 else s2.data[:n_neurons_to_show]

print(f"Sample class: {y_labels[sample_idx]}")
print(f"Layer 1 total spikes: {spk1_history.sum():.0f}")
print(f"Layer 2 total spikes: {spk2_history.sum():.0f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# --- Layer 1: Membrane Potential ---
ax = axes[0, 0]
for i in range(min(6, n_neurons_to_show)):
    ax.plot(mem1_history[:, i], alpha=0.7, linewidth=1.0)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Threshold')
ax.set_title('Layer 1 Membrane Potentials (6 neurons)')
ax.set_ylabel('Membrane V')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Layer 1: Spike Raster ---
ax = axes[0, 1]
for i in range(n_neurons_to_show):
    spike_times = np.where(spk1_history[:, i] > 0)[0]
    ax.scatter(spike_times, np.full_like(spike_times, i), s=3, c='#2c3e50')
ax.set_title('Layer 1 Spike Raster')
ax.set_ylabel('Neuron Index')
ax.set_ylim(-1, n_neurons_to_show)
ax.grid(True, alpha=0.3)

# --- Layer 2: Membrane Potential ---
ax = axes[1, 0]
for i in range(min(6, n_neurons_to_show)):
    ax.plot(mem2_history[:, i], alpha=0.7, linewidth=1.0)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Threshold')
ax.set_title('Layer 2 Membrane Potentials (6 neurons)')
ax.set_xlabel('Timestep')
ax.set_ylabel('Membrane V')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Layer 2: Spike Raster ---
ax = axes[1, 1]
for i in range(n_neurons_to_show):
    spike_times = np.where(spk2_history[:, i] > 0)[0]
    ax.scatter(spike_times, np.full_like(spike_times, i), s=3, c='#8e44ad')
ax.set_title('Layer 2 Spike Raster')
ax.set_xlabel('Timestep')
ax.set_ylabel('Neuron Index')
ax.set_ylim(-1, n_neurons_to_show)
ax.grid(True, alpha=0.3)

plt.suptitle(f'SNN Activity for Sample {sample_idx} (class {y_labels[sample_idx]})',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. ANN-to-SNN Conversion Example

A common workflow is to first train a standard ANN (with ReLU activations),
then convert it to an SNN by replacing ReLU with IF (integrate-and-fire)
neurons. The idea: ReLU(x) approximates the firing rate of an IF neuron
over many timesteps.

Steps:
1. Train an ANN with ReLU
2. Replace ReLU layers with IFNode
3. Copy the weights
4. Run the SNN for multiple timesteps and compare accuracy

In [ ]:
# Step 1: Train a small ANN
ann = nn.Sequential(
    nn.Linear(n_input, 32),
    nn.ReLU(),
    nn.Linear(32, n_output),
)

ann_optimizer = Adam(ann.parameters(), lr=1e-2)

for epoch in range(100):
    x_var = Variable(X_data)
    t_var = Variable(y_onehot)
    logits = ann(x_var)
    probs = F.softmax(logits)
    diff = probs - t_var
    loss = (diff * diff).sum() / n_samples
    loss.backward()
    ann_optimizer.step()
    ann_optimizer.zero_grad()

ann_preds = np.argmax(ann(Variable(X_data)).data, axis=1)
ann_acc = np.mean(ann_preds == y_labels) * 100
print(f"ANN accuracy: {ann_acc:.1f}%")

# Step 2: Create SNN with same architecture, replacing ReLU with IFNode
snn_fc1 = nn.Linear(n_input, 32)
snn_if1 = IFNode(threshold=1.0)
snn_fc2 = nn.Linear(32, n_output)

# Step 3: Copy weights from the trained ANN
snn_fc1.weight.data = ann.layers[0].weight.data.copy()
snn_fc1.bias.data = ann.layers[0].bias.data.copy()
snn_fc2.weight.data = ann.layers[2].weight.data.copy()
snn_fc2.bias.data = ann.layers[2].bias.data.copy()

# Step 4: Run SNN with rate coding and compare
conversion_steps = 100  # More steps = better approximation
spike_all = rate_encode(X_data, n_steps=conversion_steps)

snn_if1.reset()
snn_output = np.zeros((n_samples, n_output), dtype=np.float32)

for t in range(conversion_steps):
    x_t = Variable(spike_all[t])
    h = snn_fc1(x_t)
    s = snn_if1(h)
    out = snn_fc2(s)
    snn_output += out.data

snn_preds = np.argmax(snn_output, axis=1)
snn_acc = np.mean(snn_preds == y_labels) * 100

print(f"SNN accuracy (converted, {conversion_steps} steps): {snn_acc:.1f}%")
print(f"\nAccuracy gap: {abs(ann_acc - snn_acc):.1f}%")
print("(More timesteps generally close this gap.)")

## Summary

In this notebook you learned:

- **LIF neurons**: leaky integration + threshold + reset mechanism
- **Rate coding**: converting continuous values to spike trains
- **SNN architecture**: Linear -> LIFNode -> Linear -> LIFNode -> Linear
- **Training**: surrogate gradient approach with accumulated output
- **Visualization**: membrane potential traces and spike raster plots
- **ANN-to-SNN conversion**: replace ReLU with IFNode, copy weights

### Key Takeaways
- SNNs are inherently temporal -- they process data over multiple timesteps
- `LIFNode` provides learnable spiking dynamics with configurable decay
- `IFNode` is simpler (no leak) and useful for ANN-to-SNN conversion
- `MultiStepContainer` (not shown here) can wrap layers for automatic
  multi-timestep execution

### Next Steps
- **Notebook 04**: Vector symbolic architectures
- **Notebook 05**: Attention mechanisms and transformers